# NaN Edge Cases: Legacy vs v1

Development notebook investigating how NaN behaves across linopy operations under both conventions.

1. [shift — the primary NaN source](#shift)
2. [roll — circular, no NaN](#roll)
3. [where — conditional masking](#where)
4. [reindex — expanding coordinates](#reindex)
5. [isnull / fillna — detection and recovery](#isnull--fillna)
6. [Arithmetic with shifted expressions](#arithmetic-with-shifted-expressions)
7. [Constraints from expressions with NaN](#constraints-from-expressions-with-nan)
8. [sanitize_missings — the solver boundary](#sanitize_missings)
9. [FILL_VALUE internals](#fill_value-internals)

In [ ]:
import warnings

import pandas as pd
import xarray as xr

import linopy
from linopy import Model
from linopy.config import LinopyDeprecationWarning
from linopy.expressions import FILL_VALUE

warnings.filterwarnings("ignore", category=LinopyDeprecationWarning)

print("FILL_VALUE:", FILL_VALUE)

In [ ]:
def make_model():
    m = Model()
    time = pd.RangeIndex(5, name="time")
    x = m.add_variables(lower=0, coords=[time], name="x")
    return m, x


m, x = make_model()
print("x:", x)

---

## shift

`.shift()` is the primary structural source of NaN. It shifts data along a dimension,
creating a gap that must be filled. The fill values come from `FILL_VALUE`.

In [ ]:
expr = 2 * x + 10
shifted = expr.shift(time=1)

print("=== Original ===")
print("coeffs:", expr.coeffs.squeeze().values)
print("vars:  ", expr.vars.squeeze().values)
print("const: ", expr.const.values)

print("\n=== Shifted (time=1) ===")
print("coeffs:", shifted.coeffs.squeeze().values)
print("vars:  ", shifted.vars.squeeze().values)
print("const: ", shifted.const.values)
print("isnull:", shifted.isnull().values)

print("\nKey: vars=-1 is the integer sentinel, const=NaN marks the slot as absent.")
print("coeffs are filled with", FILL_VALUE["coeffs"], "(not NaN).")

In [ ]:
# Variables also support shift — labels get -1 sentinel, bounds get NaN
x_shifted = x.shift(time=1)
print("shifted variable labels:", x_shifted.labels.values)
print("shifted variable lower: ", x_shifted.lower.values)
print("shifted variable upper: ", x_shifted.upper.values)

---

## roll

`.roll()` is circular — values wrap around, no NaN is introduced.

In [ ]:
expr = 2 * x + 10
rolled = expr.roll(time=1)

print("=== Rolled (time=1) ===")
print("coeffs:", rolled.coeffs.squeeze().values)
print("vars:  ", rolled.vars.squeeze().values)
print("const: ", rolled.const.values)
print("isnull:", rolled.isnull().values)
print("\nNo NaN — values wrap around.")

---

## where

`.where(cond)` masks slots where the condition is False.
Masked slots get `vars=-1, coeffs=0, const=NaN` — same as FILL_VALUE.

In [ ]:
expr = 2 * x + 10
mask = xr.DataArray([True, True, False, False, True], dims=["time"])
masked = expr.where(mask)

print("=== where(mask) ===")
print("coeffs:", masked.coeffs.squeeze().values)
print("vars:  ", masked.vars.squeeze().values)
print("const: ", masked.const.values)
print("isnull:", masked.isnull().values)

print("\nFalse positions → absent slot (vars=-1, const=NaN).")
print("Same shape, fewer active slots.")

---

## reindex

`.reindex()` expands or shrinks coordinates. New coordinates get FILL_VALUE.

In [ ]:
expr = 2 * x + 10

# Expand to a larger index
new_time = pd.RangeIndex(7, name="time")
expanded = expr.reindex({"time": new_time})

print("=== reindex to [0..6] ===")
print("coeffs:", expanded.coeffs.squeeze().values)
print("vars:  ", expanded.vars.squeeze().values)
print("const: ", expanded.const.values)
print("isnull:", expanded.isnull().values)

# Shrink to a smaller index
shrunk = expr.reindex({"time": [1, 3]})
print("\n=== reindex to [1, 3] ===")
print("coeffs:", shrunk.coeffs.squeeze().values)
print("const: ", shrunk.const.values)
print("\nNew positions [5, 6] are absent. Shrinking drops slots.")

---

## isnull / fillna

`isnull()` detects absent slots. The check is:
```
(vars == -1).all(helper_dims) & const.isnull()
```
Both conditions must be true — a slot is only "absent" if there are no variables AND no constant.

In [ ]:
expr = 2 * x + 10
shifted = expr.shift(time=2)

print("=== isnull on shifted expression ===")
print("vars:  ", shifted.vars.squeeze().values)
print("const: ", shifted.const.values)
print("isnull:", shifted.isnull().values)

# What about an expression with const=0 but vars=-1?
# This would be a "zero expression" not an absent one.
print("\n=== Why const=NaN matters ===")
print("If const were 0 instead of NaN, isnull() would be False")
print("→ the slot would look like a valid 'zero expression'")
print("→ NaN in const is what distinguishes 'absent' from 'zero'")

---

## Arithmetic with shifted expressions

This is where legacy and v1 diverge. When you do arithmetic on an expression
that already has NaN (from shift/where/reindex), the NaN is **internal** — it's
not user-supplied data at an API boundary.

- **Legacy**: fills expression NaN with neutral elements before operating
- **v1**: lets IEEE NaN propagate — absent stays absent

In [ ]:
linopy.options["arithmetic_convention"] = "legacy"
m, x = make_model()

shifted = (2 * x + 10).shift(time=1)
print("=== LEGACY: shifted + 5 ===")
result = shifted + 5
print("const: ", result.const.values)
print("coeffs:", result.coeffs.squeeze().values)
print("isnull:", result.isnull().values)
print("→ NaN const filled with 0, then +5 = 5. Slot looks alive!")

print("\n=== LEGACY: shifted * 3 ===")
result = shifted * 3
print("const: ", result.const.values)
print("coeffs:", result.coeffs.squeeze().values)
print("isnull:", result.isnull().values)
print("→ NaN filled with 0, then *3 = 0. Slot has zero coeff.")

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

shifted = (2 * x + 10).shift(time=1)
print("=== V1: shifted + 5 ===")
result = shifted + 5
print("const: ", result.const.values)
print("coeffs:", result.coeffs.squeeze().values)
print("isnull:", result.isnull().values)
print("→ NaN + 5 = NaN. Absent slot stays absent. IEEE propagation.")

print("\n=== V1: shifted * 3 ===")
result = shifted * 3
print("const: ", result.const.values)
print("coeffs:", result.coeffs.squeeze().values)
print("isnull:", result.isnull().values)
print("→ NaN * 3 = NaN. Coeffs 0*3 = 0 (not NaN — coeffs FILL is 0).")

In [ ]:
# Combining expressions: absent term does NOT poison valid terms
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()
y = m.add_variables(lower=0, coords=[pd.RangeIndex(5, name="time")], name="y")

result = x * 2 + (1 * y).shift(time=1)
print("=== V1: x*2 + y.shift(1) ===")
print("const: ", result.const.values)
print("isnull:", result.isnull().values)
print("vars:")
print(result.vars.values)
print("coeffs:")
print(result.coeffs.values)
print("\n→ time=0: x[0] with coeff=2 is valid! y's absent term (vars=-1, coeffs=NaN)")
print("  does NOT mask the coordinate. const=0 (not NaN) because xr.sum skips NaN.")

### Key difference: scalar arithmetic on a single shifted expression

| | Legacy | v1 |
|---|---|---|
| `shifted + 5` at absent slot | const=5 (alive!) | const=NaN (absent) |
| `shifted * 3` at absent slot | coeffs=0, const=0 | coeffs=0, const=NaN |
| `isnull()` after arithmetic | False (slot revived!) | True (slot stays absent) |

Legacy can **revive** absent slots through scalar arithmetic. v1 cannot — once absent, always absent.

### But: combining expressions does NOT poison

When two expressions are merged (e.g., `x*2 + y.shift(1)`), each term is independent. An absent term from `y.shift` does **not** mask the valid `x` term at the same coordinate:

```python
x*2 + y.shift(time=1)  # at time=0: 2*x[0] (valid!) + absent term → 2*x[0]
```

A coordinate is only fully absent when **all** terms have `vars=-1` and `const` is NaN. This is what `isnull()` checks.

---

## Constraints from expressions with NaN

What happens when an expression with absent slots (NaN) becomes a constraint?
The NaN in const propagates to the constraint RHS.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

shifted = (1 * x).shift(time=1)
print("=== Shifted expression ===")
print("vars:  ", shifted.vars.squeeze().values)
print("const: ", shifted.const.values)

# Under v1, x[1:] - x[:-1] requires explicit join because coords differ
# (time=[1,2,3,4] vs time=[0,1,2,3]).
# Use join="override" to align by position:
print("\n=== x[1:] - x[:-1] via isel + override join ===")
x_now = 1 * x.isel(time=slice(1, None))
x_prev = 1 * x.isel(time=slice(None, -1))
ramp = x_now.sub(x_prev, join="override")
print("const: ", ramp.const.values)
print("isnull:", ramp.isnull().values)
print("→ No NaN at all — isel avoids the gap entirely.")

# But what if we use shifted expression directly as a constraint?
print("\n=== Constraint from shifted expression (has NaN) ===")
con = m.add_constraints(shifted <= 5, name="shifted_raw")
print("constraint rhs:   ", con.rhs.values)
print("constraint labels:", con.labels.values)
print("constraint vars:  ", con.vars.squeeze().values)
print("\nNaN in RHS at time=0. Label is still assigned.")
print("This will be caught by sanitize_missings() or check_has_nulls() at solve time.")

In [ ]:
# The correct approach: avoid the gap entirely with isel + override
m2, x2 = make_model()

x_now = 1 * x2.isel(time=slice(1, None))
x_prev = 1 * x2.isel(time=slice(None, -1))
ramp = x_now.sub(x_prev, join="override")
con = m2.add_constraints(ramp <= 5, name="ramp_isel")
print("=== isel + override approach (preferred) ===")
print("rhs:   ", con.rhs.values)
print("labels:", con.labels.values)
print("No NaN — constraint only exists where both operands exist.")

# Approach 2: sel with a validity mask on shifted expression
m3, x3 = make_model()
shifted = (1 * x3).shift(time=1)
valid = ~shifted.isnull()
con = m3.add_constraints(shifted.sel(time=valid) <= 5, name="shifted_sel")
print("\n=== sel approach (filter after shift) ===")
print("rhs:   ", con.rhs.values)
print("labels:", con.labels.values)
print("Absent slot at time=0 removed by .sel().")

---

## sanitize_missings

Called at solve time (before writing to solver). Sets `labels=-1` where all vars are -1.
This catches constraints where the LHS has no variables — but does NOT catch NaN in RHS.

```python
def sanitize_missings(self):
    for name in self:
        con = self[name]
        contains_non_missing = (con.vars != -1).any(con.term_dim)
        labels = self[name].labels.where(contains_non_missing, -1)
```

After sanitize_missings, `check_has_nulls()` in `.flat` catches any remaining NaN in rhs/coeffs.

In [ ]:
# Demonstrate what sanitize_missings does
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

shifted = (1 * x).shift(time=1)
# shifted at time=0: vars=-1 (no variable), const=NaN
# This means: LHS has no variables at time=0

con = m.add_constraints(shifted <= 5, name="test")
print("Before sanitize_missings:")
print("  labels:", con.labels.values)
print("  vars:  ", con.vars.squeeze().values)
print("  rhs:   ", con.rhs.values)

m.constraints.sanitize_missings()
con = m.constraints["test"]
print("\nAfter sanitize_missings:")
print("  labels:", con.labels.values)
print("  vars:  ", con.vars.squeeze().values)
print("  rhs:   ", con.rhs.values)
print("\n→ Label at time=0 set to -1 (masked out).")
print("→ RHS still has NaN but that slot is now masked by labels=-1.")

---

## FILL_VALUE internals

The sentinel values used when structural operations create absent slots:

| Type | Field | FILL_VALUE | Why |
|---|---|---|---|
| LinearExpression | `vars` | -1 | Integer sentinel (no variable) |
| LinearExpression | `coeffs` | 0 | "No term" = zero coefficient |
| LinearExpression | `const` | NaN | Marks slot as absent (needed for `isnull()`) |
| Variable | `labels` | -1 | Integer sentinel (no variable) |
| Variable | `lower` | NaN | Absent bound |
| Variable | `upper` | NaN | Absent bound |
| Constraint | `labels` | -1 | Integer sentinel (no constraint) |

### Why coeffs=0 but const=NaN?

- **coeffs=0**: A missing term contributes nothing to the sum. `0 * var = 0`.
- **const=NaN**: Distinguishes "absent slot" from "slot with zero constant."
  Without NaN in const, `isnull()` couldn't tell the difference.

### isnull() depends on const=NaN

```python
def isnull(self):
    return (self.vars == -1).all(helper_dims) & self.const.isnull()
```

If const were 0 instead of NaN, a shifted expression would not be detected as null.

In [ ]:
linopy.options["arithmetic_convention"] = "v1"
m, x = make_model()

expr = 2 * x + 10
shifted = expr.shift(time=1)

print("=== FILL_VALUE in action ===")
print(f"vars FILL={FILL_VALUE['vars']}:  ", shifted.vars.squeeze().values)
print(f"coeffs FILL={FILL_VALUE['coeffs']}: ", shifted.coeffs.squeeze().values)
print(f"const FILL={FILL_VALUE['const']}:", shifted.const.values)
print()
print("isnull:", shifted.isnull().values)
print("\nSlot 0: vars=-1, coeffs=0, const=NaN → isnull=True")
print("Slot 1: vars=0, coeffs=2, const=10 → isnull=False")

In [ ]:
linopy.options.reset()